# Efficient Data Pipelines — Scaling Beyond Basic DataLoader

**Module 32** | PyTorch Complete Learning Guide

---

This notebook covers advanced data loading patterns for production-scale training:
- IterableDataset with worker splitting
- Memory-mapped datasets
- LLM-style chunked data
- num_workers tuning
- pin_memory and prefetch_factor
- Bottleneck detection
- Bucketed batching
- DistributedSampler patterns

All examples run on **CPU only**.

## Why Data Loading Matters

In production training, the GPU often sits **idle** waiting for data:

```
Timeline (bad):   [===DATA===][GPU][===DATA===][GPU][===DATA===]
Timeline (good):  [DATA][GPU+DATA][GPU+DATA][GPU+DATA][GPU]
```

With proper pipelining, data loading overlaps with computation. Key strategies:
1. Multiple worker processes (`num_workers`)
2. Pre-fetching batches (`prefetch_factor`)
3. Pinned memory for fast GPU transfers (`pin_memory`)
4. Efficient data formats (mmap, streaming)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import (
    Dataset, IterableDataset, DataLoader,
    Sampler, DistributedSampler, get_worker_info
)
import numpy as np
import tempfile
import time
import random
import os

print(f"PyTorch version: {torch.__version__}")

## 1. IterableDataset with Worker Splitting

For streaming data (logs, web crawls, large files), `IterableDataset` avoids indexing overhead.

**Critical**: With `num_workers > 0`, each worker gets a *copy* of the dataset.
You must split the work manually to avoid duplicates.

In [ ]:
class ShardedLineDataset(IterableDataset):
    """Reads lines from multiple files, splitting across workers."""

    def __init__(self, file_paths: list):
        self.file_paths = file_paths

    def __iter__(self):
        worker_info = get_worker_info()
        if worker_info is None:
            # Single-process loading
            files = self.file_paths
        else:
            # Split files across workers
            per_worker = len(self.file_paths) // worker_info.num_workers
            start = worker_info.id * per_worker
            end = start + per_worker if worker_info.id < worker_info.num_workers - 1 else len(self.file_paths)
            files = self.file_paths[start:end]

        for fpath in files:
            with open(fpath, 'r') as f:
                for line in f:
                    tokens = [int(t) for t in line.strip().split()]
                    yield torch.tensor(tokens, dtype=torch.long)


# Create sample shard files
tmp_dir = tempfile.mkdtemp()
shard_paths = []
for i in range(4):
    path = os.path.join(tmp_dir, f"shard_{i}.txt")
    with open(path, 'w') as f:
        for _ in range(50):
            tokens = ' '.join(str(random.randint(0, 999)) for _ in range(random.randint(5, 20)))
            f.write(tokens + '\n')
    shard_paths.append(path)

dataset = ShardedLineDataset(shard_paths)
loader = DataLoader(dataset, batch_size=None, num_workers=0)

samples = [s for s in loader]
print(f"Loaded {len(samples)} sequences from 4 shards")
print(f"First sequence shape: {samples[0].shape}")
print(f"Last sequence shape: {samples[-1].shape}")

## 2. Memory-Mapped Dataset

Memory-mapped files let you access large datasets without loading everything into RAM.
The OS handles paging data in/out on demand.

**Best for**: Fixed-size records, random access, datasets larger than RAM.

In [ ]:
class MmapDataset(Dataset):
    """Memory-mapped numpy array as a PyTorch Dataset."""

    def __init__(self, path: str, shape: tuple, dtype=np.float32):
        self.mmap = np.memmap(path, dtype=dtype, mode='r', shape=shape)

    def __len__(self):
        return self.mmap.shape[0]

    def __getitem__(self, idx):
        return torch.from_numpy(self.mmap[idx].copy())


# Create a memory-mapped file with 10000 vectors of dim 128
mmap_path = os.path.join(tmp_dir, "embeddings.bin")
num_vectors = 10000
dim = 128

# Write data
fp = np.memmap(mmap_path, dtype=np.float32, mode='w+', shape=(num_vectors, dim))
fp[:] = np.random.randn(num_vectors, dim).astype(np.float32)
fp.flush()
del fp

# Read via Dataset
mmap_dataset = MmapDataset(mmap_path, shape=(num_vectors, dim))
print(f"Dataset size: {len(mmap_dataset)} vectors")
print(f"File size: {os.path.getsize(mmap_path) / 1024 / 1024:.1f} MB")
print(f"Sample shape: {mmap_dataset[0].shape}")
print(f"Random access — mmap_dataset[9999]: mean={mmap_dataset[9999].mean():.4f}")

## 3. LLM-Style Chunked Dataset

Language models train on fixed-length chunks carved from a long token stream.
This avoids padding waste entirely — every token contributes to loss.

In [ ]:
class ChunkedTokenDataset(Dataset):
    """Serves fixed-length chunks from a pre-tokenized corpus."""

    def __init__(self, token_ids: np.ndarray, chunk_size: int = 512):
        self.tokens = token_ids
        self.chunk_size = chunk_size
        self.num_chunks = len(token_ids) // chunk_size

    def __len__(self):
        return self.num_chunks

    def __getitem__(self, idx):
        start = idx * self.chunk_size
        chunk = self.tokens[start:start + self.chunk_size]
        x = torch.from_numpy(chunk[:-1].copy()).long()
        y = torch.from_numpy(chunk[1:].copy()).long()
        return x, y


# Simulate a tokenized corpus (1M tokens)
corpus = np.random.randint(0, 32000, size=1_000_000, dtype=np.int32)

chunked_dataset = ChunkedTokenDataset(corpus, chunk_size=512)
print(f"Corpus: {len(corpus):,} tokens")
print(f"Chunks: {len(chunked_dataset):,} (each {512} tokens)")

x, y = chunked_dataset[0]
print(f"Input shape: {x.shape}, Target shape: {y.shape}")
print(f"Target is input shifted by 1: {(x[1:] == y[:-1]).all()}")

## 4. num_workers Benchmark

The `num_workers` parameter controls how many subprocesses load data in parallel.

**Rules of thumb**:
- Start with `num_workers = num_cpu_cores`
- If I/O bound (slow disk): more workers help
- If memory limited: fewer workers (each gets dataset copy)
- `num_workers=0` means main process loads data (good for debugging)

In [ ]:
class SlowDataset(Dataset):
    """Simulates I/O-bound data loading."""
    def __init__(self, size=200, delay_ms=2.0):
        self.size = size
        self.delay = delay_ms / 1000.0

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        time.sleep(self.delay)
        return torch.randn(3, 64, 64), idx % 10


dataset = SlowDataset(size=100, delay_ms=2.0)

results = {}
for nw in [0, 1, 2, 4]:
    loader = DataLoader(dataset, batch_size=16, num_workers=nw)
    start = time.perf_counter()
    for batch in loader:
        pass
    elapsed = time.perf_counter() - start
    results[nw] = elapsed
    print(f"num_workers={nw}: {elapsed:.2f}s ({100/elapsed:.0f} samples/sec)")

best = min(results, key=results.get)
print(f"\nBest: num_workers={best} ({results[best]:.2f}s)")

## 5. pin_memory and prefetch_factor

### pin_memory
Allocates tensors in page-locked memory for faster GPU transfers:
```python
loader = DataLoader(..., pin_memory=True)
for data, target in loader:
    data = data.to('cuda', non_blocking=True)  # Async transfer!
```

### prefetch_factor
Number of batches each worker pre-loads ahead of time (default=2):
```python
loader = DataLoader(..., num_workers=4, prefetch_factor=4)
```
Higher values hide I/O latency but use more memory.

In [ ]:
# Demonstrate prefetch_factor effect
dataset = SlowDataset(size=80, delay_ms=3.0)

print("prefetch_factor comparison (num_workers=2):")
for pf in [1, 2, 4]:
    loader = DataLoader(
        dataset, batch_size=8, num_workers=2, prefetch_factor=pf
    )
    start = time.perf_counter()
    for batch in loader:
        time.sleep(0.01)  # Simulate fast compute
    elapsed = time.perf_counter() - start
    print(f"  prefetch_factor={pf}: {elapsed:.2f}s")

print("\npin_memory effect (CPU-only — minimal difference):")
small_dataset = SlowDataset(size=50, delay_ms=1.0)
for pin in [False, True]:
    loader = DataLoader(small_dataset, batch_size=16, pin_memory=pin)
    start = time.perf_counter()
    for batch in loader:
        pass
    elapsed = time.perf_counter() - start
    print(f"  pin_memory={pin}: {elapsed:.3f}s")

## 6. Detecting Data Loading Bottlenecks

Measure time spent waiting for data vs actual computation.
If data time dominates, your GPU is idle waiting for batches.

**Pattern:**
```python
for batch in loader:
    data_time = time.time() - end  # Time since last batch
    # ... compute ...
    compute_time = time.time() - start
    end = time.time()
```

In [ ]:
def profile_data_pipeline(loader, num_batches=10):
    """Profile data loading vs compute time."""
    data_times = []
    compute_times = []

    end = time.perf_counter()
    for i, (data, target) in enumerate(loader):
        if i >= num_batches:
            break
        data_times.append(time.perf_counter() - end)

        # Simulate compute
        start = time.perf_counter()
        _ = data.mean()
        time.sleep(0.02)
        compute_times.append(time.perf_counter() - start)

        end = time.perf_counter()

    avg_data = sum(data_times) / len(data_times)
    avg_compute = sum(compute_times) / len(compute_times)
    return avg_data, avg_compute


# Data-bound scenario
slow_dataset = SlowDataset(size=200, delay_ms=10.0)
slow_loader = DataLoader(slow_dataset, batch_size=16, num_workers=0)

avg_d, avg_c = profile_data_pipeline(slow_loader)
print("Data-bound scenario (num_workers=0, slow I/O):")
print(f"  Avg data time:    {avg_d*1000:.1f}ms")
print(f"  Avg compute time: {avg_c*1000:.1f}ms")
print(f"  Data fraction:    {avg_d/(avg_d+avg_c)*100:.0f}% ← BOTTLENECK")

# Fixed with workers
fast_loader = DataLoader(slow_dataset, batch_size=16, num_workers=2)
avg_d2, avg_c2 = profile_data_pipeline(fast_loader)
print(f"\nWith num_workers=2:")
print(f"  Avg data time:    {avg_d2*1000:.1f}ms")
print(f"  Avg compute time: {avg_c2*1000:.1f}ms")
print(f"  Data fraction:    {avg_d2/(avg_d2+avg_c2)*100:.0f}%")

## 7. Bucketed Batching for Variable-Length Sequences

Standard batching pads all sequences to the longest in the batch — wasteful.

**Bucketed batching** groups similar-length sequences together:
- Sort by length
- Divide into buckets
- Sample batches from within buckets
- Shuffle bucket order for randomness

In [ ]:
class VariableLengthDataset(Dataset):
    def __init__(self, size=500, min_len=5, max_len=200):
        self.sequences = [
            torch.randn(random.randint(min_len, max_len), 32)
            for _ in range(size)
        ]

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]


class BucketBatchSampler(Sampler):
    """Groups similar-length sequences into batches."""

    def __init__(self, lengths, batch_size, num_buckets=10):
        sorted_idx = sorted(range(len(lengths)), key=lambda i: lengths[i])
        bucket_size = len(sorted_idx) // num_buckets

        self.batches = []
        for b in range(num_buckets):
            start = b * bucket_size
            end = start + bucket_size if b < num_buckets - 1 else len(sorted_idx)
            bucket = sorted_idx[start:end]
            random.shuffle(bucket)
            for i in range(0, len(bucket), batch_size):
                self.batches.append(bucket[i:i + batch_size])
        random.shuffle(self.batches)

    def __iter__(self):
        return iter(self.batches)

    def __len__(self):
        return len(self.batches)


def collate_pad(batch):
    max_len = max(s.size(0) for s in batch)
    padded = torch.zeros(len(batch), max_len, batch[0].size(-1))
    lengths = []
    for i, s in enumerate(batch):
        padded[i, :s.size(0)] = s
        lengths.append(s.size(0))
    return padded, torch.tensor(lengths)


var_dataset = VariableLengthDataset(size=500)
lengths = [len(var_dataset[i]) for i in range(len(var_dataset))]

# Random batching
random_loader = DataLoader(var_dataset, batch_size=16, shuffle=True, collate_fn=collate_pad)
random_waste = sum(
    (p.size(1) * len(l) - l.sum().item()) * p.size(2)
    for p, l in random_loader
)

# Bucketed batching
bucket_sampler = BucketBatchSampler(lengths, batch_size=16)
bucket_loader = DataLoader(var_dataset, batch_sampler=bucket_sampler, collate_fn=collate_pad)
bucket_waste = sum(
    (p.size(1) * len(l) - l.sum().item()) * p.size(2)
    for p, l in bucket_loader
)

print(f"Random batching — padding elements: {random_waste:,.0f}")
print(f"Bucket batching — padding elements: {bucket_waste:,.0f}")
print(f"Reduction: {(1 - bucket_waste/random_waste)*100:.1f}% less padding waste")

## 8. DistributedSampler Pattern

For multi-GPU training, each process must see a **unique subset** of data.
`DistributedSampler` handles this:

```python
sampler = DistributedSampler(
    dataset,
    num_replicas=world_size,  # Total GPU count
    rank=rank,                 # This GPU's index
    shuffle=True,
    drop_last=True,           # Even batch sizes across GPUs
)
loader = DataLoader(dataset, sampler=sampler, batch_size=32)

for epoch in range(num_epochs):
    sampler.set_epoch(epoch)  # CRITICAL: ensures different shuffle per epoch
    for batch in loader:
        ...
```

In [ ]:
# Simulate DistributedSampler behavior (single process demo)
dataset = list(range(100))

class SimpleDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

ds = SimpleDataset(list(range(100)))

# Simulate 4 GPUs
world_size = 4
print(f"Dataset size: {len(ds)}, World size: {world_size}")
print(f"Each GPU sees: {len(ds) // world_size} samples\n")

for rank in range(world_size):
    sampler = DistributedSampler(ds, num_replicas=world_size, rank=rank, shuffle=True)
    sampler.set_epoch(0)
    indices = list(sampler)
    print(f"  GPU {rank}: {len(indices)} samples, first 5 indices = {indices[:5]}")

# Verify no overlap
all_indices = []
for rank in range(world_size):
    sampler = DistributedSampler(ds, num_replicas=world_size, rank=rank, shuffle=False)
    all_indices.extend(list(sampler))

print(f"\nTotal indices across all GPUs: {len(all_indices)}")
print(f"Unique indices: {len(set(all_indices))}")
print(f"No overlap: {len(set(all_indices)) == len(all_indices)}")

## 9. Combining Patterns: Production DataLoader

A production-grade DataLoader configuration brings it all together:

In [ ]:
def create_production_dataloader(
    dataset,
    batch_size=32,
    num_workers=4,
    is_distributed=False,
    rank=0,
    world_size=1,
):
    """Creates an optimized DataLoader for production training."""
    sampler = None
    shuffle = True

    if is_distributed:
        sampler = DistributedSampler(
            dataset, num_replicas=world_size, rank=rank,
            shuffle=True, drop_last=True
        )
        shuffle = False  # Sampler handles shuffling

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        prefetch_factor=2 if num_workers > 0 else None,
        persistent_workers=num_workers > 0,
        drop_last=True,
    )
    return loader


# Demo
demo_ds = SimpleDataset(list(range(1000)))
loader = create_production_dataloader(demo_ds, batch_size=32, num_workers=2)
print(f"Production loader created:")
print(f"  Batches: {len(loader)}")
print(f"  num_workers: {loader.num_workers}")
print(f"  pin_memory: {loader.pin_memory}")
print(f"  persistent_workers: {loader.persistent_workers}")
print(f"  drop_last: {loader.drop_last}")

# Verify it works
batch = next(iter(loader))
print(f"  First batch size: {len(batch)}")

## 10. Exercise: Chunked CSV IterableDataset

**Task**: Create an `IterableDataset` that reads a large CSV file in chunks.

Requirements:
1. Read the CSV in chunks of N rows (don't load the whole file)
2. Support multi-worker loading (split chunks across workers)
3. Yield individual rows as tensors

Hint: Use `pandas.read_csv(..., chunksize=N)` or manual line reading.

In [ ]:
# Exercise: Implement ChunkedCSVDataset

class ChunkedCSVDataset(IterableDataset):
    """Reads a CSV file in chunks, splitting across workers."""

    def __init__(self, filepath: str, chunk_size: int = 100):
        self.filepath = filepath
        self.chunk_size = chunk_size
        # Count total lines for worker splitting
        with open(filepath) as f:
            self.total_lines = sum(1 for _ in f) - 1  # Exclude header

    def __iter__(self):
        worker_info = get_worker_info()
        if worker_info is None:
            start_line = 0
            end_line = self.total_lines
        else:
            per_worker = self.total_lines // worker_info.num_workers
            start_line = worker_info.id * per_worker
            end_line = start_line + per_worker
            if worker_info.id == worker_info.num_workers - 1:
                end_line = self.total_lines

        with open(self.filepath) as f:
            header = next(f)  # Skip header
            for i, line in enumerate(f):
                if i < start_line:
                    continue
                if i >= end_line:
                    break
                values = [float(v) for v in line.strip().split(',')]
                yield torch.tensor(values, dtype=torch.float32)


# Create sample CSV
csv_path = os.path.join(tmp_dir, "large_data.csv")
num_cols = 10
num_rows = 1000
with open(csv_path, 'w') as f:
    f.write(','.join(f'col_{i}' for i in range(num_cols)) + '\n')
    for _ in range(num_rows):
        f.write(','.join(f'{random.gauss(0, 1):.4f}' for _ in range(num_cols)) + '\n')

# Test
csv_dataset = ChunkedCSVDataset(csv_path, chunk_size=100)
loader = DataLoader(csv_dataset, batch_size=32)

total = 0
for batch in loader:
    total += batch.size(0)

print(f"CSV file: {num_rows} rows x {num_cols} columns")
print(f"Loaded {total} samples via chunked reading")
print(f"Batch shape: {batch.shape}")

## Key Takeaways

| Pattern | When to Use | Benefit |
|---------|-------------|--------|
| `IterableDataset` | Streaming/huge data, no random access | No index overhead, natural sharding |
| Memory-mapped files | Large fixed-size datasets | OS-managed caching, low RAM |
| Chunked tokens | LLM pre-training | Zero padding waste |
| `num_workers > 0` | Always for training | Overlaps I/O with compute |
| `pin_memory=True` | GPU training | Faster host→device transfer |
| Bucketed batching | Variable-length sequences | Less padding, faster training |
| `DistributedSampler` | Multi-GPU training | Even data distribution |
| `persistent_workers` | Repeated epochs | Avoids worker restart cost |

### Performance Checklist

1. **Profile first** — measure data time vs compute time
2. **Tune `num_workers`** — start with CPU core count
3. **Enable `pin_memory`** for GPU training
4. **Use `persistent_workers=True`** to avoid restart overhead
5. **Increase `prefetch_factor`** if compute-bound
6. **Bucket by length** for NLP/audio data
7. **Use memory-mapping** for datasets larger than RAM
8. **Stream data** when random access isn't needed

In [ ]:
# Cleanup
import shutil
shutil.rmtree(tmp_dir, ignore_errors=True)
print("Notebook complete! Cleaned up temp files.")